# 06 — N64 reconstruction: 64-bubble GW spectrum comparison

Compare the 3D sledgehamr run (64 bubbles, L=200, λ̄=0.84) with the
spectrum reconstructed from the 1D BM scan × collision weights.

Two comparison times:
- **T₁**: last available SH snapshot
- **T₂**: time when all pair weights drop to zero (last collision)

**Workflow**
Run the notebook from top to bottom. Section 1 builds the weights binary and
regenerates the periodic N=64 collision weights automatically.

In [1]:
import glob
import os
import subprocess
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

import ptbuilder as pt
from ptbuilder.physics import InstantonProfile, BubbleKinematics
from ptbuilder.analysis import write_weights_input, load_weights, apply_keffsq
from ptbuilder.scan import load_scan
from ptbuilder.ic import _cutoff_times

# 64 bubble positions from reference_codes/notebooks/Scan.ipynb
bubbles = np.array([[  0.        ,   0.        ,   0.        ],
       [104.21080818, 134.00331875, 115.32260336],
       [ 82.91633086,  40.22040859, 161.10457444],
       [162.20889478, 164.79584079,   2.8384039 ],
       [154.47733984, 187.65793396,  20.16866321],
       [199.13096101,  27.01104231,  45.30013382],
       [168.5003722 ,  38.64441063,  52.81041635],
       [133.97360158,  91.59693268, 138.34643773],
       [ 50.28297449,  51.95537184,  23.62294384],
       [177.13769118, 134.66889799,   8.71668886],
       [ 30.68289591,   9.45788771,  50.50424856],
       [ 64.12769861, 195.12306707,  81.57164046],
       [ 56.68936652, 130.84497227, 108.2354709 ],
       [ 82.99174886, 121.03920058, 172.37415331],
       [127.09687502, 120.51398912, 134.53558472],
       [ 33.36926752,  13.63720659,   0.60996262],
       [ 18.39375397, 191.40492629, 122.47329764],
       [ 19.63641252,  63.09685344, 174.17147189],
       [ 90.74169878, 154.13735512, 104.39306946],
       [ 86.557716  ,  72.91515918, 182.95789915],
       [119.33310673,  37.76924713, 149.28485556],
       [ 17.13480987, 132.62322674, 189.63031283],
       [109.46636148, 118.08537517,  76.31000499],
       [170.09845239,   3.4854559 ,  23.50245978],
       [ 18.70476325, 147.66938079,  58.80734523],
       [111.64277024, 157.053698  , 132.24005744],
       [ 42.14101276,  50.93855746, 176.75407925],
       [ 14.51198083,  16.93867057,  60.2360491 ],
       [ 52.77856703,  29.30614115,  36.71599   ],
       [ 38.07656718,  13.10864189,  72.07363165],
       [138.51107657, 177.90126384, 114.77664924],
       [ 25.84809123, 112.82413992,  33.27661835],
       [  6.6733563 , 160.69190438, 139.70034561],
       [197.70475434,  13.8230847 ,  60.37034021],
       [ 12.25179549, 100.55314862,  78.92943814],
       [140.93499401, 164.76389065, 124.56483354],
       [100.40316917,  68.76075668,  17.56098047],
       [144.15542294,  50.57241845,  83.47993277],
       [ 43.42498771,  75.67311377, 122.44988237],
       [ 73.56937364,  38.65136447,   0.68298585],
       [160.40914341,   1.50148649,   2.225444  ],
       [127.8706212 ,  29.43082072,  23.094503  ],
       [ 10.24110325, 111.83791255, 156.45308871],
       [161.49404006, 163.38664412, 161.98093237],
       [ 27.04229405,  45.36589381, 117.35604037],
       [126.85079112, 162.26356355,  70.60492779],
       [128.66689612,  68.71766065, 150.35375572],
       [166.89624473, 124.99140447,  49.97804878],
       [ 22.58882999, 172.17683864,  41.74842148],
       [173.1491633 , 103.35193697,  79.45686904],
       [ 63.47467018, 157.89290094,  54.88476635],
       [ 83.264726  , 141.90140818, 151.48688241],
       [ 71.99367711,  16.40398225, 197.67281455],
       [ 55.41434491, 148.76063021,   3.30900276],
       [136.79240841,  60.8792625 , 193.43780416],
       [ 21.14423948,  15.16246438, 137.40606156],
       [ 37.08759453,  55.1292117 ,  75.06349727],
       [  6.48071698, 123.73465632,   4.00908165],
       [ 49.63963798,  63.59991319, 136.27840358],
       [  6.47400652, 161.05000329, 163.66077338],
       [142.56695796, 110.77450728, 194.29867779],
       [ 85.21456401,  72.43223496, 118.71280211],
       [103.95645911,   4.65524665, 154.29805374],
       [151.90410329, 147.74780743,  72.22549058]])

In [ ]:
REPO_ROOT   = Path(pt.__file__).parent.parent
SH_RUNS_DIR = Path('/pscratch/sd/b/buschman/other_runs/')
DATA_DIR    = REPO_ROOT / 'data'

lb         = 0.84
L          = 200.
zero_pad   = 1.
N_WEIGHTS  = 1024
T_SCAN_MAX = 70.
N_SCAN     = 32

WEIGHTS_IN  = DATA_DIR / 'weights_in_N64.h5'
WEIGHTS_OUT = DATA_DIR / 'weights_out_N64.h5'
SH_PATH     = SH_RUNS_DIR / 'bubbles_N64'

scan_times    = np.linspace(1., T_SCAN_MAX, N_SCAN)
weights_times = np.linspace(1., T_SCAN_MAX, N_WEIGHTS)


## 1. Set up weights calculation

Writes the HDF5 input file for the `cpp/weights` binary and generates a shell script.

In [3]:
def load_kinematics_local(lb):
    """Load BubbleKinematics from stored instanton.h5 (no C++ needed)."""
    path = DATA_DIR / f'phi4_lambda_bar{lb}' / 'instanton.h5'
    with h5py.File(path, 'r') as f:
        profile = InstantonProfile(
            R=f['R'][:], Phi=f['Phi'][:],
            rin_0 =float(f.attrs['rin_0']),
            rmid_0=float(f.attrs['rmid_0']),
            rout_0=float(f.attrs['rout_0']),
            interp=None,
        )
    return BubbleKinematics(profile)


kin = load_kinematics_local(lb)

write_weights_input(
    path=WEIGHTS_IN,
    positions=bubbles,
    t=weights_times,
    kinematics=kin,
    L=L,
)
print(f'Written: {WEIGHTS_IN.name}')

weights_dir = REPO_ROOT / 'cpp/weights'
weights_bin = weights_dir / 'weights'
subprocess.run(['make'], cwd=weights_dir, check=True)
result = subprocess.run(
    [str(weights_bin), str(WEIGHTS_IN), str(WEIGHTS_OUT)],
    check=True, capture_output=True, text=True,
)
print(result.stdout)

Written: weights_in_N64.h5

Run: bash scripts/run_weights_N64.sh


## 2. Load scan results

In [4]:
cache = DATA_DIR / f'phi4_lambda_bar{lb}' / 'scan_cache.h5'
scan_results = load_scan(cache)
actual_t = next(iter(scan_results.values())).times
print(f'lb={lb}: {len(scan_results)} gamma values  '
      f't=[{actual_t[0]:.1f}, {actual_t[-1]:.1f}]  n_t={len(actual_t)}')

lb=0.84: 32 gamma values  t=[1.0, 70.0]  n_t=32


## 3. Load collision weights

The corrected periodic weights were generated automatically in Section 1.

In [5]:
if not WEIGHTS_OUT.exists():
    print('Weights not found — rerun Section 1')
else:
    weights, pair_i, pair_j, gammas = load_weights(WEIGHTS_OUT, N_WEIGHTS)
    n_pairs = len(pair_i)
    print(f'{n_pairs} colliding pairs')

    pair_d  = np.zeros(n_pairs)
    pair_tm = np.zeros(n_pairs)
    for p in range(n_pairs):
        i, j = int(pair_i[p]), int(pair_j[p])
        dp = bubbles[i] - bubbles[j]
        dp -= L * np.round(dp / L)
        d = float(np.linalg.norm(dp))
        pair_d[p]  = d
        pair_tm[p] = float(_cutoff_times(d, 0, 1.)[2])   # BM comparison time ≈ 1.095*d

    print(f'd     range: [{pair_d.min():.2f}, {pair_d.max():.2f}]')
    print(f'gamma range: [{gammas.min():.4f}, {gammas.max():.4f}]')
    print(f't_m   range: [{pair_tm.min():.2f}, {pair_tm.max():.2f}]')

328 colliding pairs
d     range: [16.56, 133.02]
gamma range: [1.0799, 9.5776]
t_m   range: [18.13, 145.66]


## 4. Load sledgehamr 3D spectrum

In [6]:
def _list_gw_files(sh_path):
    pattern = os.path.join(sh_path, 'gw_spectra', '*', 'spectra.hdf5')
    entries = []
    for fp in glob.glob(pattern):
        dname = os.path.basename(os.path.dirname(fp))
        if dname.isdigit():
            entries.append((int(dname), fp))
    entries.sort(key=lambda x: x[0])
    return entries


def _read_gw_file(fp):
    with h5py.File(fp, 'r') as f:
        k_int = np.asarray(f['k' if 'k' in f else 'k_sq'][:])
        spec  = np.asarray(f['Spectrum' if 'Spectrum' in f else 'spectrum'][:])
        hdr   = np.asarray(f['Header'][:]).reshape(-1)
        t     = float(hdr[0])
    return apply_keffsq(k_int), spec, t


def load_sh_all_snaps(sh_path):
    files = _list_gw_files(sh_path)
    if not files:
        raise FileNotFoundError(f'No gw_spectra/*/spectra.hdf5 in {sh_path}')
    snaps = [_read_gw_file(fp) for _, fp in files]
    snaps.sort(key=lambda x: x[2])
    return snaps


def normalize_sh(k_sq, spectrum, L, zero_pad):
    if k_sq[0] == 0:
        k_sq     = k_sq[1:]
        spectrum = spectrum[1:]
    Leff = L * zero_pad
    r    = np.sqrt(k_sq)
    k    = r * (2 * np.pi / Leff)
    norm = 4.0 * (2 * np.pi * r) * Leff**3
    return k, spectrum * norm


sh_snaps = load_sh_all_snaps(SH_PATH)
t0 = sh_snaps[0][2]
t1 = sh_snaps[-1][2]
print(f'{len(sh_snaps)} snapshots, t=[{t0:.2f}, {t1:.2f}]')

46 snapshots, t=[0.00, 90.01]


## 5. Reconstruct and compare

In [ ]:
from scipy.interpolate import RegularGridInterpolator


def build_log_scan_interpolator(scan_results, k_out):
    gammas = np.array(sorted(scan_results.keys()))
    times  = next(iter(scan_results.values())).times
    Ng, Nt, Nk = len(gammas), len(times), len(k_out)

    spec_grid = np.zeros((Ng, Nt, Nk))
    for ig, gamma in enumerate(gammas):
        res = scan_results[gamma]
        for it in range(Nt):
            spec_grid[ig, it] = np.interp(k_out, res.w, res.spectrum[it],
                                           left=0., right=0.)

    floor     = spec_grid.max() * 1e-12
    log_grid  = np.log(np.maximum(spec_grid, floor))
    log_floor = np.log(floor)

    _raw = RegularGridInterpolator(
        (gammas, times), log_grid,
        bounds_error=False, fill_value=log_floor,
    )

    def interp_fn(pts):
        return np.exp(_raw(pts))

    return interp_fn, gammas, times, k_out


def bm_reconstruct(gamma, weights_t, weights_times, t_max, interp_fn, times, gammas):
    """
    Reconstruct dE/dlnk from the 1D BM scan for one pair.

    Steps at the FINE weights_times resolution, not the coarse (32-point)
    scan grid: the collision weight can vary much faster in time (brief
    collision windows) than the spectrum (smooth, slowly-varying in log), so
    the reconstruction must be time-stepped at the weights' resolution, with
    the spectrum freshly interpolated from the scan at each fine step --
    the scan exists specifically to be interpolated, not to define the
    reconstruction's own time grid. Matches
    reference_codes/notebooks/Scan.ipynb's combined_spectrum loop.
    """
    g = float(np.clip(gamma, gammas.min(), gammas.max()))

    t_hi   = min(t_max, times[-1])
    mask   = (weights_times >= times[0]) & (weights_times <= t_hi)
    t_comp = weights_times[mask]
    if len(t_comp) == 0 or t_comp[-1] < t_hi:
        t_comp = np.append(t_comp, t_hi)

    w = np.interp(t_comp, weights_times, weights_t)

    pts   = np.column_stack([np.full(len(t_comp), g), t_comp])
    specs = interp_fn(pts)

    delta = np.diff(specs, axis=0)
    return (w[:len(t_comp) - 1, None] * delta).sum(axis=0)


def interp_sh_at_t(snaps, t_target):
    ts = np.array([s[2] for s in snaps])
    if t_target <= ts[0]:
        return snaps[0][0], snaps[0][1], ts[0]
    if t_target >= ts[-1]:
        return snaps[-1][0], snaps[-1][1], ts[-1]
    i = int(np.searchsorted(ts, t_target)) - 1
    k_sq, s_a, t_a = snaps[i]
    _,    s_b, t_b = snaps[i + 1]
    alpha = (t_target - t_a) / (t_b - t_a)
    floor = max(s_a.max(), s_b.max()) * 1e-12
    log_a = np.log(np.maximum(s_a, floor))
    log_b = np.log(np.maximum(s_b, floor))
    return k_sq, np.exp(log_a + alpha * (log_b - log_a)), t_target

In [ ]:
# Comparison times.
# Weights are 0 before AND after each collision (they peak during the event).
# T2 = last time any pair is still actively colliding (sum_w > threshold).
T1 = sh_snaps[-1][2]  # last snapshot — upper limit

sum_w  = weights.sum(axis=0)            # (n_weights,) — summed over all pairs
active = np.where(sum_w > 0.01)[0]
T2     = float(weights_times[active[-1]]) if len(active) else weights_times[-1]

print(f'T1 = {T1:.2f}  (last snapshot)')
print(f'T2 = {T2:.2f}  (last collision complete)')

# Build scan interpolator
first_res = next(iter(scan_results.values()))
k_out = first_res.w
interp_fn, gammas_grid, scan_t, _ = build_log_scan_interpolator(scan_results, k_out)

# BM reconstruction: bm_reconstruct's t_hi = min(t_max, scan_t[-1]) clamps
# T1 and T2 to the same effective bound (scan_t[-1] = T_SCAN_MAX), and
# weights_times never exceeds T_SCAN_MAX either -- so the reconstruction has
# already fully saturated by T2 (weight ~0 for every pair beyond it) and
# recon(T1) carries no new physics over recon(T2). Compute once at T2 and
# verify that assumption instead of computing (and plotting) both.
recon_T2 = np.zeros(len(k_out))
for p in range(n_pairs):
    recon_T2 += bm_reconstruct(float(gammas[p]), weights[p], weights_times, T2,
                                interp_fn, scan_t, gammas_grid)

recon_T1 = np.zeros(len(k_out))
for p in range(n_pairs):
    recon_T1 += bm_reconstruct(float(gammas[p]), weights[p], weights_times, T1,
                                interp_fn, scan_t, gammas_grid)
assert np.allclose(recon_T1, recon_T2, rtol=1e-6, atol=recon_T2.max() * 1e-9), \
    'recon(T1) and recon(T2) diverge -- the "no new physics beyond T2" assumption may not hold here'
print('Verified: recon(T1) == recon(T2) (no new physics between T2 and T1, as expected)')
recon = recon_T2

# 3D SH spectra at T1 and T2 -- these ARE genuinely different (real snapshots)
sh_at = {}
for T in [T1, T2]:
    k_sq, spec, t_act = interp_sh_at_t(sh_snaps, T)
    k_sh, y_sh = normalize_sh(k_sq, spec, L, zero_pad)
    sh_at[T] = (k_sh, y_sh, t_act)

fig, ax = plt.subplots(figsize=(8, 5))

k1, y1, ta1 = sh_at[T1]
k2, y2, ta2 = sh_at[T2]

# 3D SH: shaded band between T2 (lower) and T1 (upper)
y2_r = np.interp(k1, k2, y2, left=0., right=0.)
ax.fill_between(k1, y2_r, y1, color='C0', alpha=0.15, linewidth=0)
ax.plot(k2, y2, color='C0', lw=2, linestyle=':', label=rf'3+1D  $t={ta2:.0f}$ (last collision)')
ax.plot(k1, y1, color='C0', lw=2,              label=rf'3+1D  $t={ta1:.0f}$ (last snapshot)')

# 1+1D reconstruction — single curve (see note above: T1 and T2 give the same result)
ax.plot(k_out, recon, color='black', lw=2, linestyle='--', label='1+1D incoherent recon')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e3,6e7)
ax.set_xlim(0.02,20)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
ax.legend(frameon=False)
ax.set_title(rf'N64 reconstruction — $\bar{{\lambda}}={lb}$')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_reconstruction.pdf', bbox_inches='tight')
plt.show()

## 6. Coherent amplitude reconstruction

Extends the N=3 coherent-summing approach (notebook 07) to the full N=64 pair
set (328 colliding pairs). Running 328 individual per-pair BubbleMaster
amplitude jobs isn't practical, so instead this interpolates the complex
amplitude A(w, cos_theta, t) from the shared (γ, time) `scan_cache.h5` table
at each pair's own (fractional) γ — the same table now carries amplitude
data once built with `--save-amplitude` (notebook 01).

Same convention as notebook 07: at each time step, pairs are summed
*coherently* (with position phase factors, before squaring); the resulting
per-step power is then summed *incoherently* over time.

Requires `scan_cache.h5` to have been (re)built with amplitude — if you see
a "no amplitude data" error below, the notebook 01 scan job hasn't finished
yet (or was run before `--save-amplitude` was added).

In [ ]:
first_gamma_check = next(iter(scan_results.values()))
if first_gamma_check.amp_re is None:
    raise RuntimeError(
        'scan_cache.h5 has no amplitude data yet -- rerun notebook 01 '
        '(--save-amplitude is already wired into its SLURM scripts) and '
        'wait for the scan job to finish before running this section.'
    )

# gammas_grid_amp fixes a deterministic gamma order (ascending) for
# everything in this section -- in particular, first_res below must be the
# SAME gamma group that build_amp_interpolator resamples onto (see below),
# not whatever HDF5/dict iteration happens to return first.
gammas_grid_amp = np.array(sorted(scan_results.keys()))
first_res = scan_results[gammas_grid_amp[0]]

k_amp = first_res.k
n_w_amp, n_k_amp = first_res.amp_re.shape[1:]
w_amp = first_res.w
print(f'Amplitude grid: n_w={n_w_amp}  n_k={n_k_amp}  '
      f'cos_theta=[{k_amp[0]:.3f},{k_amp[-1]:.3f}]')


def build_amp_interpolator(scan_results, w_out):
    """(gamma, time) -> (amp_re, amp_im), each (n_w, n_k), linearly interpolated.

    Unlike the power spectrum (build_log_scan_interpolator above), amplitude
    isn't positive-definite, so this interpolates linearly rather than in
    log-space.

    IMPORTANT: each gamma's own `w` grid is NOT the same array -- BubbleMaster
    derives wmin from that pair's spatial grid, which depends on gamma via
    the bubble separation d (see ptbuilder/ic.py's write_2d_setup: wmin =
    pi/((n_z-1)*dz)/2, wmax fixed). Confirmed on real scan data: wmin varies
    ~5x across the gamma range while wmax stays fixed. So every gamma's
    amp_re/amp_im must be resampled onto a single common `w_out` grid before
    being stacked into this (gamma,time,w,k) table -- otherwise the same
    array index `iw` would refer to a different physical frequency depending
    on gamma, corrupting the (gamma,time) bilinear interpolation below. This
    mirrors the resampling build_log_scan_interpolator already does for the
    power-only spectrum (np.interp(k_out, res.w, ...)), just applied to the
    complex amplitude instead.
    """
    grid_gammas = np.array(sorted(scan_results.keys()))
    grid_times  = next(iter(scan_results.values())).times
    Ng, Nt      = len(grid_gammas), len(grid_times)
    n_w, n_k    = len(w_out), scan_results[grid_gammas[0]].amp_re.shape[2]

    re_grid = np.zeros((Ng, Nt, n_w, n_k))
    im_grid = np.zeros((Ng, Nt, n_w, n_k))
    for ig, g in enumerate(grid_gammas):
        res = scan_results[g]
        if res.amp_re is None:
            raise RuntimeError(f'gamma={g} has no amplitude data in scan_cache.h5')
        if np.array_equal(res.w, w_out):
            re_grid[ig] = res.amp_re
            im_grid[ig] = res.amp_im
        else:
            for it in range(Nt):
                for ik in range(n_k):
                    re_grid[ig, it, :, ik] = np.interp(w_out, res.w, res.amp_re[it, :, ik],
                                                        left=0., right=0.)
                    im_grid[ig, it, :, ik] = np.interp(w_out, res.w, res.amp_im[it, :, ik],
                                                        left=0., right=0.)

    flat_shape = (Ng, Nt, n_w * n_k)
    _raw_re = RegularGridInterpolator((grid_gammas, grid_times), re_grid.reshape(flat_shape),
                                      bounds_error=False, fill_value=0.)
    _raw_im = RegularGridInterpolator((grid_gammas, grid_times), im_grid.reshape(flat_shape),
                                      bounds_error=False, fill_value=0.)

    def interp_fn(pts):
        re = _raw_re(pts).reshape(-1, n_w, n_k)
        im = _raw_im(pts).reshape(-1, n_w, n_k)
        return re, im

    return interp_fn, grid_gammas, grid_times


amp_interp_fn, gammas_grid_amp, scan_t_amp = build_amp_interpolator(scan_results, w_amp)
print(f'Amplitude interpolator built: {len(gammas_grid_amp)} gammas x {len(scan_t_amp)} times')

In [ ]:
from numpy.polynomial.legendre import leggauss

# Pair axes and center-of-mass positions (minimum-image PBC) for all colliding
# pairs -- same formula as notebook 07's N=3 version, generalized to n_pairs.
pair_axes    = {}
pair_centers = {}
for p in range(n_pairs):
    i, j = int(pair_i[p]), int(pair_j[p])
    dp = bubbles[i] - bubbles[j]
    dp -= L * np.round(dp / L)
    pair_axes[p]    = dp / np.linalg.norm(dp)
    pair_centers[p] = (0.5 * (bubbles[i] + bubbles[j])) % L

# Gauss-Legendre x uniform-phi quadrature on the sphere (same as notebook 07)
N_THETA, N_PHI = 64, 128
cos_nodes, gl_w = leggauss(N_THETA)
phi_vals = np.linspace(0., 2.*np.pi, N_PHI, endpoint=False)
d_phi    = 2.*np.pi / N_PHI

CT, PH = np.meshgrid(cos_nodes, phi_vals, indexing='ij')
ST     = np.sqrt(np.maximum(1. - CT**2, 0.))
hat_k  = np.stack([ST*np.cos(PH), ST*np.sin(PH), CT], axis=-1).reshape(-1, 3)
d_omega = np.outer(gl_w, np.full(N_PHI, d_phi)).ravel()

N_sph = len(hat_k)
print(f'Sphere quadrature: {N_sph} points')

# Precompute per-pair direction-dependent factors once (time-independent) --
# cos_th[p] : (N_sph,) cos(theta) relative to this pair's own axis
# cp/sp[p]  : (n_w_amp, N_sph) cos/sin of the position phase w*(k_hat . center)
cos_th = {p: np.clip(np.abs(hat_k @ pair_axes[p]), 0., 1.) for p in range(n_pairs)}
ph_ang = {p: np.outer(w_amp, hat_k @ pair_centers[p]) for p in range(n_pairs)}
cp_ang = {p: np.cos(ph_ang[p]) for p in range(n_pairs)}
sp_ang = {p: np.sin(ph_ang[p]) for p in range(n_pairs)}
print(f'Precomputed direction factors for {n_pairs} pairs')

In [ ]:
def active_pairs_for(t_max, threshold=0.01):
    """Skip pairs whose weight never exceeds `threshold` within [1, t_max] --
    pure efficiency filter (same threshold convention already used for T2
    above)."""
    mask_t = weights_times <= t_max
    return [p for p in range(n_pairs) if np.any(weights[p][mask_t] > threshold)]


def _amp_at_times(g_arr, t_pts):
    """Query the (gamma, time) amplitude interpolator for a set of pairs
    (given their gammas g_arr) at shared time points t_pts.
    Returns re, im with shape (len(g_arr), len(t_pts), n_w, n_k)."""
    n_p, n_t = len(g_arr), len(t_pts)
    pts = np.column_stack([np.repeat(g_arr, n_t), np.tile(t_pts, n_p)])
    re, im = amp_interp_fn(pts)
    return re.reshape(n_p, n_t, n_w_amp, n_k_amp), im.reshape(n_p, n_t, n_w_amp, n_k_amp)


def coherent_sum_n64(t_max, pair_batch=32, t_chunk=32):
    """
    Coherent sum across all active pairs, per time step (with position
    phase factors, before squaring), then incoherent sum over time -- same
    formula as notebook 07's coherent_sum, generalized to n_pairs.

    Stepped at the FINE weights_times resolution (not the coarse 32-point
    scan grid): matches dA_pair_series in notebook 07 -- the amplitude scan
    is smooth/slowly-varying in time and exists to be interpolated at each
    fine step (collisions can be brief and change the weight much faster
    than the amplitude varies), not to define the reconstruction's own time
    grid.

    The fine grid can have ~1000 points and there are up to ~330 active
    pairs, so amplitude values for the *entire* fine grid are never held in
    memory at once for all pairs (that was the ~30GB blowup risk) -- instead
    `t_chunk` fine intervals are interpolated, diffed, and consumed at a
    time. `pair_batch` separately bounds the pair-axis batch size for the
    direction-interpolation + phase-multiply step (as before, via
    np.take_along_axis instead of a per-pair Python loop). Peak memory ~
    n_active * t_chunk * n_w * n_k, independent of the total fine-grid
    length.
    """
    import time as _time
    active   = active_pairs_for(t_max)
    n_active = len(active)
    g_arr    = np.clip(gammas[active], gammas_grid_amp.min(), gammas_grid_amp.max())

    t_hi   = min(t_max, scan_t_amp[-1])
    mask   = (weights_times >= scan_t_amp[0]) & (weights_times <= t_hi)
    t_grid = weights_times[mask]
    if len(t_grid) == 0 or t_grid[-1] < t_hi:
        t_grid = np.append(t_grid, t_hi)
    n_t = len(t_grid)   # fine grid points; n_t-1 diff intervals total

    cos_th_stack = np.stack([cos_th[p] for p in active])   # (n_active, N_sph)
    cp_stack     = np.stack([cp_ang[p] for p in active])   # (n_active, n_w, N_sph)
    sp_stack     = np.stack([sp_ang[p] for p in active])   # (n_active, n_w, N_sph)

    idx  = np.clip(np.searchsorted(k_amp, cos_th_stack), 1, n_k_amp - 1)   # (n_active, N_sph)
    frac = (cos_th_stack - k_amp[idx-1]) / (k_amp[idx] - k_amp[idx-1])     # (n_active, N_sph)
    idx_lo3 = np.broadcast_to((idx-1)[:, None, :], (n_active, n_w_amp, N_sph))
    idx_hi3 = np.broadcast_to(idx[:, None, :],     (n_active, n_w_amp, N_sph))
    frac3   = frac[:, None, :]

    t_start = _time.time()
    print(f'  t_max={t_max:.1f}: {n_active}/{n_pairs} pairs active, {n_t} fine time steps')

    P_coh = np.zeros(n_w_amp)
    for c0 in range(0, n_t - 1, t_chunk):
        c1    = min(c0 + t_chunk, n_t - 1)   # chunk covers diff intervals [c0, c1)
        t_pts = t_grid[c0:c1 + 1]            # c1-c0+1 grid points -> c1-c0 diffs

        re_c, im_c = _amp_at_times(g_arr, t_pts)   # (n_active, n_pts, n_w, n_k)
        dar_c = np.diff(re_c, axis=1)               # (n_active, n_pts-1, n_w, n_k)
        dai_c = np.diff(im_c, axis=1)

        wp_c = np.sqrt(np.clip(
            np.stack([np.interp(t_pts[:-1], weights_times, weights[p]) for p in active]),
            0., None))                               # (n_active, n_pts-1)
        dar_c = dar_c * wp_c[:, :, None, None]
        dai_c = dai_c * wp_c[:, :, None, None]

        for it in range(dar_c.shape[1]):
            A_tot_re = np.zeros((n_w_amp, N_sph))
            A_tot_im = np.zeros((n_w_amp, N_sph))
            for lo in range(0, n_active, pair_batch):
                hi = min(lo + pair_batch, n_active)

                re_lo = np.take_along_axis(dar_c[lo:hi, it], idx_lo3[lo:hi], axis=2)
                re_hi = np.take_along_axis(dar_c[lo:hi, it], idx_hi3[lo:hi], axis=2)
                im_lo = np.take_along_axis(dai_c[lo:hi, it], idx_lo3[lo:hi], axis=2)
                im_hi = np.take_along_axis(dai_c[lo:hi, it], idx_hi3[lo:hi], axis=2)
                f = frac3[lo:hi]

                Ap_re = re_lo * (1 - f) + re_hi * f   # (batch, n_w, N_sph)
                Ap_im = im_lo * (1 - f) + im_hi * f

                cp_b = cp_stack[lo:hi]
                sp_b = sp_stack[lo:hi]
                A_tot_re += (Ap_re * cp_b - Ap_im * sp_b).sum(axis=0)
                A_tot_im += (Ap_re * sp_b + Ap_im * cp_b).sum(axis=0)

            P_coh += ((A_tot_re**2 + A_tot_im**2) * d_omega[None, :]).sum(axis=-1)

        print(f'    fine steps {c0+1}-{c1}/{n_t-1}: {_time.time()-t_start:.1f} s elapsed', flush=True)

    print(f'  t_max={t_max:.1f} done: {_time.time()-t_start:.1f} s total')
    return P_coh * w_amp**3 * 2. * np.pi

In [ ]:
CPP_COHERENT_OUT = REPO_ROOT / 'cpp_out_full.h5'
if not CPP_COHERENT_OUT.exists():
    raise FileNotFoundError(
        f'{CPP_COHERENT_OUT} not found -- run on the cluster (from the repo '
        'root, ideally under a short interactive allocation):\n'
        f'  bin/coherent {WEIGHTS_IN} {WEIGHTS_OUT} {cache} {CPP_COHERENT_OUT.name} '
        f'--t-max {T_SCAN_MAX}'
    )

with h5py.File(CPP_COHERENT_OUT) as f:
    w_cpp     = f['w'][:]
    recon_coh = f['P_coh'][:].reshape(int(f.attrs['n_t_max']), int(f.attrs['n_w']))[0]

# Single run at t_max=T_SCAN_MAX -- same "T1 carries no new physics beyond
# T2" reasoning as the incoherent case above (compute_coherent_spectrum
# clamps t_max the same way bm_reconstruct does).
print(f'Coherent reconstruction loaded from {CPP_COHERENT_OUT.name}  '
      f'(single run at t_max={T_SCAN_MAX})')

# --- Reproduce the same comparison plot as Section 5, with the coherent curve added ---
fig, ax = plt.subplots(figsize=(8, 5))

# 3D SH: shaded band between T2 (lower) and T1 (upper) — same as before
y2_r = np.interp(k1, k2, y2, left=0., right=0.)
ax.fill_between(k1, y2_r, y1, color='C0', alpha=0.15, linewidth=0)
ax.plot(k2, y2, color='C0', lw=2, linestyle=':', label=rf'3+1D  $t={ta2:.0f}$ (last collision)')
ax.plot(k1, y1, color='C0', lw=2,              label=rf'3+1D  $t={ta1:.0f}$ (last snapshot)')

# 1+1D reconstructions — single curves (T1/T2 give the same result, see Section 5)
ax.plot(k_out, recon,     color='black', lw=2, linestyle='--', label='1+1D incoherent')
ax.plot(w_cpp, recon_coh, color='C2',    lw=2, linestyle='-.', label='1+1D coherent')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(1e3,6e7)
ax.set_xlim(0.02,20)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$\mathrm{d}E_{\mathrm{GW}}/\mathrm{d}\ln k$')
ax.legend(frameon=False, fontsize=8)
ax.set_title(rf'N64 reconstruction — $\bar{{\lambda}}={lb}$  (coherent vs incoherent)')
plt.tight_layout()
plt.savefig(REPO_ROOT / 'n64_reconstruction_coherent.pdf', bbox_inches='tight')
plt.show()

print(f"\nPeak values:")
print(f"  incoherent: {recon.max():.3e}   coherent: {recon_coh.max():.3e}")

In [28]:
np.exp(328)

2.8092478959838913e+142

## 7. Validation reference for `cpp/coherent`

`coherent_sum_n64` is too slow in Python to run to completion at N=64, so it
was ported to C++/OpenMP (`cpp/coherent`). This cell saves a small,
fast-to-compute reference (restricted to a small `t_max`) to compare against
the C++ binary's output -- see `scripts/validate_coherent_cpp.py`.

In [ ]:
import time as _time

# Small enough to finish in a couple of minutes -- adjust if too slow/fast
# (coherent_sum_n64 prints per-chunk timing, use that to judge).
t_max_test = 5.0

_t0 = _time.time()
active_test = active_pairs_for(t_max_test)
P_coh_test = coherent_sum_n64(t_max_test)
print(f'validation reference: t_max={t_max_test}  n_active={len(active_test)}  '
      f'wall time={_time.time()-_t0:.1f}s')

np.savez(REPO_ROOT / 'data' / 'validation_coherent_reference.npz',
         w=w_amp, P_coh=P_coh_test, t_max=t_max_test, n_active=len(active_test))
print('Saved data/validation_coherent_reference.npz')